# Módulo 02 · Aula 03 — Desfazendo Alterações

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

Esta é a aula que separa quem *usa* Git de quem *confia* no Git.

Quando você sabe desfazer qualquer coisa, para de ter medo de experimentar. E é aí que a ferramenta começa a valer o que promete.

## O mapa de decisão

Antes dos comandos, a pergunta certa: **onde está a coisa que você quer desfazer?**

```
                    ┌─────────────────────────────────┐
                    │  O que você quer desfazer?      │
                    └────────────┬────────────────────┘
                                 │
         ┌───────────────────────┼───────────────────────┐
         ▼                       ▼                       ▼
  Mudança no arquivo      Arquivo no stage         Um commit
  (ainda não no stage)                                   │
         │                       │              ┌────────┴────────┐
         ▼                       ▼              ▼                 ▼
  git restore <arq>    git restore --staged   Já foi           Só local
                              <arq>          empurrado?
                                                 │                 │
                                                 ▼                 ▼
                                            git revert        git reset
                                            (seguro)         (reescreve)

         ┌──────────────────────────────────────────────┐
         │  Quer só GUARDAR para depois? → git stash    │
         │  Perdeu alguma coisa?         → git reflog   │
         └──────────────────────────────────────────────┘
```

## O que você vai aprender aqui

| # | Comando | Desfaz o quê |
|---|---------|--------------|
| 1 | `git restore` | Mudanças em arquivos |
| 2 | `git commit --amend` | O último commit (só o último) |
| 3 | `git revert` | Um commit, criando um commit inverso |
| 4 | `git reset` | Move o branch para trás |
| 5 | `git stash` | Guarda temporariamente |
| 6 | `git reflog` | 🆘 Recupera o que parecia perdido |
| 7 | `git cherry-pick` | Traz um commit específico |

## ⚙️ Preparando o laboratório

In [ ]:
import shutil
import subprocess
from pathlib import Path

BASE = Path("lab_git_03").resolve()
shutil.rmtree(BASE, ignore_errors=True)
REPO = BASE / "atlas"
REPO.mkdir(parents=True)


def git(*args, cwd=REPO, mostrar=True):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if mostrar:
        print("$ git " + " ".join(args))
        saida = (r.stdout + r.stderr).rstrip()
        print(saida if saida else "(sem saída)")
        print()
    return r


def escrever(nome, conteudo, pasta=REPO):
    caminho = Path(pasta) / nome
    caminho.parent.mkdir(parents=True, exist_ok=True)
    caminho.write_text(conteudo, encoding="utf-8")


def ler(nome, pasta=REPO):
    return (Path(pasta) / nome).read_text(encoding="utf-8")


def estado():
    """Resumo compacto do estado atual — usaremos muito."""
    print("─" * 60)
    git("log", "--oneline", "-5", mostrar=False)
    r = subprocess.run(["git", "log", "--oneline", "-5"], cwd=REPO,
                       capture_output=True, text=True)
    print("HISTÓRICO:")
    print("  " + (r.stdout.strip().replace("\n", "\n  ") or "(vazio)"))
    r = subprocess.run(["git", "status", "--short"], cwd=REPO,
                       capture_output=True, text=True)
    print("STATUS:")
    print("  " + (r.stdout.strip().replace("\n", "\n  ") or "(limpo)"))
    print("─" * 60)


git("init", "-b", "main", mostrar=False)
git("config", "user.name", "Aluno Atlas", mostrar=False)
git("config", "user.email", "aluno@aurora.com.br", mostrar=False)

escrever("config.py", 'TAXA_IMPOSTO = 0.18\nFRETE_GRATIS = 500.0\n')
git("add", "."); git("commit", "-m", "chore: adiciona configuração", mostrar=False)

escrever("metricas.py", '''"""Métricas de vendas."""


def faturamento(vendas):
    """Soma os pedidos pagos."""
    return sum(v["qtd"] * v["preco"] for v in vendas if v["status"] == "pago")
''')
git("add", "."); git("commit", "-m", "feat: adiciona cálculo de faturamento", mostrar=False)

escrever("README.md", "# Atlas\n\nRelatórios da Aurora Comércio.\n")
git("add", "."); git("commit", "-m", "docs: adiciona README", mostrar=False)

estado()
print("📁 Laboratório:", BASE)

## 1. `git restore` — desfazendo mudanças em arquivos

Este é o comando do dia a dia. Ele **não** mexe no histórico — só nos arquivos.

| Comando | Desfaz |
|---------|--------|
| `git restore arquivo` | Mudanças no working directory (volta ao último commit) |
| `git restore --staged arquivo` | Tira do stage (mantém a mudança no arquivo) |
| `git restore --staged --worktree arquivo` | Ambos: tira do stage **e** descarta |
| `git restore --source=abc123 arquivo` | Traz a versão de um commit específico |
| `git restore .` | Tudo, a partir da pasta atual |

> 🔴 **`git restore arquivo` descarta seu trabalho permanentemente.** Não há histórico de mudanças não commitadas. O Git não pode recuperar o que ele nunca viu. Confirme com `git diff` antes.

In [ ]:
# Situação 1: editei e me arrependi (ainda não dei add)
escrever("metricas.py", ler("metricas.py") + "\n# rascunho que não presta\nimport os  # nem uso\n")

print(">>> O que mudou:")
git("diff", "--stat")
git("status", "--short")

In [ ]:
git("restore", "metricas.py")
print(">>> Depois do restore:")
git("status", "--short")
print(">>> Conteúdo do arquivo:")
print(ler("metricas.py"))

In [ ]:
# Situação 2: dei add e me arrependi (quero tirar do stage, mas MANTER a edição)
escrever("config.py", 'TAXA_IMPOSTO = 0.20\nFRETE_GRATIS = 500.0\nMOEDA = "BRL"\n')
git("add", "config.py")
git("status", "--short")

In [ ]:
git("restore", "--staged", "config.py")
print(">>> Saiu do stage, mas a edição continua no arquivo:")
git("status", "--short")
print(ler("config.py"))

In [ ]:
# Situação 3: tirar do stage E descartar a edição, de uma vez
git("restore", "--staged", "--worktree", "config.py")
git("status", "--short")
print(">>> Voltou ao estado do último commit:")
print(ler("config.py"))

> 💡 **Leitura do `git status --short`:** são duas colunas. A primeira é o **stage**, a segunda é o **working directory**.
>
> | Símbolo | Significado |
> |---------|-------------|
> | `M ` | Modificado **e** no stage |
> | ` M` | Modificado, **fora** do stage |
> | `MM` | No stage, e modificado de novo depois |
> | `A ` | Adicionado (arquivo novo, no stage) |
> | `??` | Untracked |
> | `D ` | Deletado |

## 2. `git commit --amend` — corrigindo o último commit

Serve para dois casos muito comuns:

1. **Errei a mensagem** do commit
2. **Esqueci um arquivo** no commit

```bash
git commit --amend -m "mensagem corrigida"     # só a mensagem
git add arquivo-esquecido
git commit --amend --no-edit                   # adiciona o arquivo, mantém a mensagem
```

> ⚠️ **`--amend` não edita o commit — ele cria um novo e joga o antigo fora.** O hash muda. Por isso:
>
> 🔴 **Só use `--amend` em commits que você ainda NÃO enviou (`push`).** Se o commit já está no servidor, alterá-lo cria divergência com todo mundo que já baixou.

In [ ]:
# Errei a mensagem
escrever("formatacao.py", '''def formatar_brl(valor):
    """Formata como R$ 1.234,56."""
    return "R$ " + f"{valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
''')
git("add", "."); git("commit", "-m", "aserta formatacao", mostrar=False)
git("log", "--oneline", "-2")

In [ ]:
git("commit", "--amend", "-m", "feat(formatacao): adiciona formatação monetária brasileira")
git("log", "--oneline", "-2")

> 📌 Compare os hashes antes e depois. **São diferentes.** O commit antigo não foi editado — foi substituído. (Ele ainda existe no `reflog`, como veremos na seção 6.)

In [ ]:
# Esqueci um arquivo
escrever("testes_formatacao.py", '''from formatacao import formatar_brl

assert formatar_brl(1234.5) == "R$ 1.234,50"
print("✅ ok")
''')

git("add", "testes_formatacao.py")
git("commit", "--amend", "--no-edit")
git("log", "--oneline", "--stat", "-1")

## 3. `git revert` — desfazendo com segurança

**`revert` cria um commit NOVO que desfaz o que outro commit fez.**

```
ANTES:   A ◀── B ◀── C ◀── D          (D introduziu um bug)

DEPOIS:  A ◀── B ◀── C ◀── D ◀── D'   (D' desfaz D)
```

O histórico **cresce**. Nada é apagado. Por isso:

> ✅ **`revert` é a forma correta de desfazer algo que já foi para o servidor.**

Todo mundo que já baixou o commit D continua tendo o commit D — e recebe o D' que o anula. Ninguém fica com histórico divergente.

In [ ]:
# Um commit que quebra a produção
escrever("metricas.py", '''"""Métricas de vendas."""


def faturamento(vendas):
    """Soma TODOS os pedidos."""
    # BUG: removeu o filtro de status — cancelados entram na conta!
    return sum(v["qtd"] * v["preco"] for v in vendas)
''')
git("add", "."); git("commit", "-m", "perf: simplifica cálculo de faturamento", mostrar=False)
git("log", "--oneline", "-3")

print(">>> O código com bug:")
print(ler("metricas.py"))

In [ ]:
# Descobrimos o problema. Como já foi para produção, revertemos.
git("revert", "HEAD", "--no-edit")
git("log", "--oneline", "-4")

In [ ]:
print(">>> O código voltou ao estado correto:")
print(ler("metricas.py"))

### Variações do `revert`

```bash
git revert HEAD                    # desfaz o último
git revert abc1234                 # desfaz um commit específico
git revert HEAD~2                  # desfaz o antepenúltimo
git revert abc1234..def5678        # desfaz um intervalo
git revert -n abc1234              # prepara, mas NÃO commita (--no-commit)
git revert -m 1 <commit-de-merge>  # desfaz um merge (o -m diz qual pai manter)
git revert --abort                 # 🆘 cancela um revert conflitado
```

> 💡 Revert também pode dar conflito — se o código mudou muito desde o commit revertido. Resolve-se igual a um merge: edite, `git add`, `git revert --continue`.

## 4. `git reset` — movendo o branch para trás

**`reset` move o ponteiro do branch.** Os commits à frente ficam órfãos (mas recuperáveis pelo reflog por algumas semanas).

```
ANTES:   A ◀── B ◀── C ◀── D
                             ▲
                            main

git reset --hard B:

         A ◀── B ◀── C ◀── D    ← C e D viram órfãos
               ▲
              main
```

### Os três modos — a tabela mais importante desta aula

| Modo | Move o branch | Staging area | Working directory | Perde código? |
|------|:-------------:|:------------:|:-----------------:|:-------------:|
| `--soft` | ✅ | mantém | mantém | ❌ Não |
| `--mixed` (padrão) | ✅ | limpa | mantém | ❌ Não |
| `--hard` | ✅ | limpa | **sobrescreve** | 🔴 **SIM** |

Traduzindo:

- **`--soft`** — "desfaz o commit, mas deixa tudo pronto para commitar de novo"
- **`--mixed`** — "desfaz o commit e tira do stage; os arquivos continuam editados"
- **`--hard`** — "faz de conta que nada aconteceu" 💣

> 🔴 **`--hard` destrói trabalho não commitado, e isso não tem volta.** O reflog recupera *commits*, não recupera edições que nunca viraram commit.
>
> **Regra de segurança:** antes de qualquer `--hard`, rode `git status`. Se houver algo não commitado que importa, faça `git stash` primeiro.

In [ ]:
# Três commits pequenos que deveriam ter sido um só
escrever("relatorio.py", "def gerar():\n    pass\n")
git("add", "."); git("commit", "-m", "wip", mostrar=False)

escrever("relatorio.py", "def gerar(vendas):\n    linhas = []\n    return linhas\n")
git("add", "."); git("commit", "-m", "wip 2", mostrar=False)

escrever("relatorio.py", '''def gerar(vendas):
    """Monta o relatório."""
    linhas = ["RELATÓRIO"]
    for v in vendas:
        linhas.append(f"{v['cidade']}: {v['valor']}")
    return "\\n".join(linhas)
''')
git("add", "."); git("commit", "-m", "wip 3 agora vai", mostrar=False)

git("log", "--oneline", "-5")

In [ ]:
# --soft: desfaz os 3 commits, mas mantém tudo no stage
git("reset", "--soft", "HEAD~3")
git("log", "--oneline", "-3")
git("status", "--short")

In [ ]:
# Um único commit limpo no lugar dos três
git("commit", "-m", "feat(relatorio): adiciona geração de relatório em texto")
git("log", "--oneline", "-3")

> 💡 **Esse é o uso mais valioso do `reset --soft`:** transformar vários commits bagunçados (`wip`, `wip 2`, `agora vai`) em um commit único e bem descrito, **antes** de abrir o Pull Request. Ninguém precisa ver sua bagunça.

In [ ]:
# --mixed (padrão): desfaz o commit e tira do stage
escrever("temp.py", "# arquivo temporário\n")
git("add", "."); git("commit", "-m", "chore: adiciona arquivo temporário", mostrar=False)

git("reset", "HEAD~1")        # sem flag = --mixed
git("status", "--short")
print(">>> O arquivo continua no disco?", (REPO / "temp.py").exists())

In [ ]:
# --hard: apaga tudo. Note que 'temp.py' era untracked — o --hard NÃO remove untracked.
git("reset", "--hard", "HEAD")
git("status", "--short")
print(">>> temp.py ainda existe?", (REPO / "temp.py").exists(),
      " <- --hard não toca em arquivos untracked")

In [ ]:
# Para limpar untracked existe o git clean
git("clean", "-n")            # -n = dry run: mostra o que APAGARIA
git("clean", "-f")            # -f = force: apaga de verdade
print(">>> temp.py agora?", (REPO / "temp.py").exists())

### `git clean` — removendo arquivos não rastreados

| Comando | O que faz |
|---------|-----------|
| `git clean -n` | **Simula** — só lista o que seria apagado |
| `git clean -f` | Apaga arquivos untracked |
| `git clean -fd` | Apaga também **pastas** untracked |
| `git clean -fdx` | Apaga inclusive o que está no `.gitignore` (💣 leva o `.venv`!) |

> 🔴 **Sempre rode `git clean -n` antes de `git clean -f`.** É um comando sem volta.

### ⚠️ `reset` vs `revert` — a decisão

| | `git reset` | `git revert` |
|---|-------------|--------------|
| Histórico | Reescreve (apaga commits) | Cresce (adiciona commit) |
| Seguro em branch compartilhado? | 🔴 **Não** | ✅ Sim |
| Precisa de `push --force`? | Sim | Não |
| Quando usar | Commits **locais**, ainda não enviados | Qualquer coisa **já enviada** |

**A regra prática:**

> Se você já deu `push`, use **`revert`**.
> Se ainda é só seu, pode usar **`reset`**.

## 5. `git stash` — a gaveta

*"Estou no meio de uma coisa e preciso trocar de branch agora."*

`stash` guarda suas mudanças não commitadas em uma pilha e deixa o diretório limpo.

| Comando | O que faz |
|---------|-----------|
| `git stash` | Guarda mudanças rastreadas |
| `git stash -u` | Guarda também os **untracked** |
| `git stash -m "descrição"` | Guarda com um nome |
| `git stash list` | Lista a pilha |
| `git stash show -p` | Mostra o diff do último |
| `git stash pop` | Aplica **e remove** da pilha |
| `git stash apply` | Aplica e **mantém** na pilha |
| `git stash apply stash@{2}` | Aplica um específico |
| `git stash drop` | Descarta o último |
| `git stash clear` | 💣 Esvazia a pilha inteira |
| `git stash branch nome` | Cria um branch a partir do stash |

In [ ]:
# No meio de uma refatoração...
escrever("metricas.py", '''"""Métricas de vendas — REFATORAÇÃO EM ANDAMENTO."""

from collections import defaultdict


def faturamento(vendas):
    """Soma os pedidos pagos."""
    return sum(v["qtd"] * v["preco"] for v in vendas if v["status"] == "pago")


def por_cidade(vendas):
    # TODO: ainda não terminei isso
    pass
''')
escrever("rascunho_ideias.txt", "ideias para a refatoração\n- separar em classes?\n")

git("status", "--short")

In [ ]:
# Chega a urgência: preciso do branch main limpo AGORA
git("stash", "-u", "-m", "refatoração de métricas em andamento")
git("status", "--short")
git("stash", "list")

In [ ]:
# Resolvo a urgência com o diretório limpo
escrever("config.py", 'TAXA_IMPOSTO = 0.18\nFRETE_GRATIS = 500.0\nTIMEOUT_API = 30\n')
git("add", "."); git("commit", "-m", "fix: adiciona timeout na configuração da API", mostrar=False)
git("log", "--oneline", "-2")

In [ ]:
# Volto para onde estava
git("stash", "pop")
git("status", "--short")
print(">>> O rascunho voltou?", (REPO / "rascunho_ideias.txt").exists())
print(">>> A refatoração voltou?", "REFATORAÇÃO" in ler("metricas.py"))

In [ ]:
# Limpando a bagunça do exemplo
git("restore", "metricas.py", mostrar=False)
(REPO / "rascunho_ideias.txt").unlink()
git("status", "--short")

> 💡 **`pop` vs `apply`:** `pop` remove da pilha depois de aplicar. `apply` mantém. Use `apply` quando quiser aplicar o mesmo stash em mais de um branch, ou quando não tiver certeza de que vai dar certo.
>
> ⚠️ **Stash não é backup.** É uma gaveta temporária, local, que não vai para o servidor e que você vai esquecer. Se o trabalho é importante, faça um commit num branch de rascunho — é mais seguro e você consegue empurrar para o remoto.

## 6. 🆘 `git reflog` — a rede de segurança

**O reflog registra todo movimento do `HEAD`.** Cada commit, troca de branch, merge, reset, rebase — tudo fica registrado, mesmo que os commits tenham ficado órfãos.

Isso significa:

> **Quase nada se perde de verdade no Git**, desde que já tenha virado commit alguma vez.

O reflog é **local** (não vai para o servidor) e os registros expiram em ~90 dias.

In [ ]:
git("reflog", "-12")

### Como ler o reflog

```
a3f5c9e HEAD@{0}: commit: fix: adiciona timeout
b2c4d8a HEAD@{1}: reset: moving to HEAD~3
c1d3e7b HEAD@{2}: commit: wip 3 agora vai
```

- **`HEAD@{0}`** — onde você está agora
- **`HEAD@{1}`** — o passo anterior
- **`HEAD@{n}`** — n passos atrás

Para voltar: `git reset --hard HEAD@{2}` ou use o hash direto.

In [ ]:
# 💣 O DESASTRE: reset --hard destruindo commits
git("log", "--oneline", "-4")
print(">>> Vamos apagar os 3 últimos commits com --hard...")
git("reset", "--hard", "HEAD~3")
git("log", "--oneline", "-3")

In [ ]:
# 😱 "Perdi o trabalho de hoje!"
# 🧘 Não perdeu. Olhe o reflog:
git("reflog", "-8")

In [ ]:
# A recuperação: volte para o ponto ANTES do reset
r = subprocess.run(["git", "reflog", "--format=%H %gs"], cwd=REPO,
                   capture_output=True, text=True)
linhas = r.stdout.strip().splitlines()

# Acha a entrada imediatamente anterior ao reset
alvo = None
for i, linha in enumerate(linhas):
    if "reset: moving to HEAD~3" in linha:
        alvo = linhas[i + 1].split()[0]
        break

print("Commit a recuperar:", alvo[:7])
git("reset", "--hard", alvo)
git("log", "--oneline", "-4")
print("✅ Recuperado.")

### O protocolo de emergência

Quando algo der errado, **nesta ordem**:

1. **Respire.** Não rode mais comandos no impulso.
2. `git status` — onde estou?
3. `git reflog` — onde eu estava?
4. `git reset --hard HEAD@{n}` — volta para lá
5. Se preferir não mexer no branch atual: `git switch -c recuperacao HEAD@{n}`

> ⚠️ **O que o reflog NÃO recupera:**
> - Mudanças que nunca viraram commit (perdidas por `restore` ou `reset --hard`)
> - Arquivos apagados por `git clean`
> - Arquivos que nunca foram adicionados ao Git
>
> Moral: **commite cedo, commite com frequência.** Um commit feio é infinitamente melhor que trabalho perdido. Você limpa o histórico depois com `reset --soft`.

## 7. `git cherry-pick` — pegando um commit específico

Aplica **um commit** de outro branch no branch atual, sem trazer o resto.

```
       A ◀── B ◀── C              ← main
        \
         D ◀── E ◀── F            ← feature

git switch main; git cherry-pick E

       A ◀── B ◀── C ◀── E'       ← main (só o E, recriado)
```

**Casos de uso:**

- Um hotfix feito no branch errado
- Uma correção pontual que precisa ir para `main` antes da feature inteira
- Recuperar um commit órfão encontrado no reflog

In [ ]:
git("switch", "-c", "feature/experimental", mostrar=False)

escrever("experimento.py", "# código experimental que não vai para main\n")
git("add", "."); git("commit", "-m", "feat: experimento", mostrar=False)

escrever("config.py", ler("config.py").replace("TIMEOUT_API = 30", "TIMEOUT_API = 60"))
git("add", "."); git("commit", "-m", "fix: aumenta timeout da API para 60s", mostrar=False)

escrever("experimento.py", "# mais coisa experimental\n")
git("add", "."); git("commit", "-m", "feat: mais experimentos", mostrar=False)

git("log", "--oneline", "-4")

In [ ]:
# Só a correção do timeout precisa ir para main, agora
r = subprocess.run(["git", "log", "--oneline", "--grep=timeout", "-1", "--format=%H"],
                   cwd=REPO, capture_output=True, text=True)
hash_fix = r.stdout.strip()

git("switch", "main", mostrar=False)
git("cherry-pick", hash_fix)
git("log", "--oneline", "-3")
print(">>> O timeout foi para 60?", "TIMEOUT_API = 60" in ler("config.py"))
print(">>> O experimento veio junto?", (REPO / "experimento.py").exists())

> ⚠️ **Cherry-pick cria um commit NOVO** (hash diferente) com o mesmo conteúdo. Se depois você mesclar o branch inteiro, o Git geralmente percebe que a mudança já está lá — mas pode conflitar. Use cherry-pick com parcimônia; ele não substitui um merge.

## 8. Tabela de decisão — a cola que resolve 95% dos casos

| Situação | Comando |
|----------|---------|
| Editei um arquivo e me arrependi | `git restore arquivo` |
| Dei `add` sem querer | `git restore --staged arquivo` |
| Quero descartar **tudo** que não commitei | `git restore .` |
| Errei a mensagem do último commit (não enviado) | `git commit --amend -m "..."` |
| Esqueci um arquivo no último commit (não enviado) | `git add arq && git commit --amend --no-edit` |
| Quero desfazer um commit **já enviado** | `git revert <hash>` |
| Quero juntar meus últimos 3 commits em um | `git reset --soft HEAD~3 && git commit` |
| Quero desfazer o último commit local, mantendo o código | `git reset HEAD~1` |
| Quero apagar o último commit local e o código | `git reset --hard HEAD~1` ⚠️ |
| Preciso trocar de branch no meio de algo | `git stash` → depois `git stash pop` |
| Apaguei arquivos novos sem querer | 🔴 Sem volta se nunca foram commitados |
| Fiz `reset --hard` e me arrependi | `git reflog` → `git reset --hard HEAD@{n}` |
| Quero um commit específico de outro branch | `git cherry-pick <hash>` |
| Merge deu ruim e quero cancelar | `git merge --abort` |
| Quero limpar arquivos untracked | `git clean -n` → `git clean -f` |
| Quero voltar um arquivo à versão de 3 commits atrás | `git restore --source=HEAD~3 arquivo` |

## 🔧 Prática guiada — Cinco desastres e suas soluções

Cada célula é um cenário completo: o problema, o diagnóstico e a correção.

In [ ]:
# Repositório limpo para os cenários
shutil.rmtree(BASE / "desastres", ignore_errors=True)
D = BASE / "desastres"
D.mkdir(parents=True)


def gd(*args, mostrar=True):
    return git(*args, cwd=D, mostrar=mostrar)


def ed(nome, conteudo):
    escrever(nome, conteudo, pasta=D)


gd("init", "-b", "main", mostrar=False)
gd("config", "user.name", "Aluno Atlas", mostrar=False)
gd("config", "user.email", "aluno@aurora.com.br", mostrar=False)

for i in range(1, 4):
    ed(f"modulo_{i}.py", f"# módulo {i}\n")
    gd("add", ".", mostrar=False)
    gd("commit", "-m", f"feat: adiciona módulo {i}", mostrar=False)

gd("log", "--oneline")

In [ ]:
# ══════════════════════════════════════════════════════════
# DESASTRE 1 — "Commitei a senha do banco"
# ══════════════════════════════════════════════════════════
ed(".env", "DB_PASSWORD=SuperSecreta2026!\nAPI_KEY=sk-abc123xyz\n")
gd("add", ".", mostrar=False)
gd("commit", "-m", "chore: adiciona configuração", mostrar=False)
print("😱 A senha está no histórico:")
gd("show", "--stat", "HEAD")

In [ ]:
# SOLUÇÃO (ainda NÃO foi enviado ao servidor):
gd("reset", "--soft", "HEAD~1")           # desfaz o commit
gd("restore", "--staged", ".env")         # tira do stage
ed(".gitignore", ".env\n__pycache__/\n")
gd("add", ".gitignore", mostrar=False)
gd("commit", "-m", "chore: adiciona .gitignore", mostrar=False)

print("✅ Resolvido. Arquivos versionados:")
gd("ls-files")
print(">>> O .env ainda existe no disco?", (D / ".env").exists())

> 🔴 **Se o commit JÁ tivesse ido para o GitHub**, este procedimento não bastaria. A ordem correta seria:
>
> 1. **Trocar a senha imediatamente.** Antes de qualquer coisa técnica.
> 2. Depois, se ainda fizer sentido, reescrever o histórico com `git filter-repo` ou o BFG Repo-Cleaner.
> 3. Forçar o push e avisar todo o time para re-clonar.
>
> O passo 1 é o único que realmente importa. Segredo exposto é segredo queimado.

In [ ]:
# ══════════════════════════════════════════════════════════
# DESASTRE 2 — "Trabalhei 2 horas no branch errado"
# ══════════════════════════════════════════════════════════
ed("nova_feature.py", "# feature nova, trabalho de 2 horas\ndef algo_importante():\n    pass\n")
gd("add", ".", mostrar=False)
gd("commit", "-m", "feat: adiciona feature importante", mostrar=False)
print("😱 Commitei em main, mas devia estar num branch:")
gd("log", "--oneline", "-2")

In [ ]:
# SOLUÇÃO: cria o branch a partir daqui, depois recua o main
gd("branch", "feature/nova-funcionalidade")   # branch aponta para o commit atual
gd("reset", "--hard", "HEAD~1")               # main volta um commit

print(">>> main:")
gd("log", "--oneline", "-2")
print(">>> feature/nova-funcionalidade:")
gd("log", "--oneline", "-2", "feature/nova-funcionalidade")
print("✅ O trabalho está salvo no branch certo.")

In [ ]:
# ══════════════════════════════════════════════════════════
# DESASTRE 3 — "Fiz reset --hard e perdi tudo"
# ══════════════════════════════════════════════════════════
gd("switch", "feature/nova-funcionalidade", mostrar=False)
for i in range(1, 4):
    ed(f"parte_{i}.py", f"# trabalho importante parte {i}\n")
    gd("add", ".", mostrar=False)
    gd("commit", "-m", f"feat: parte {i} da funcionalidade", mostrar=False)

gd("log", "--oneline", "-4")
print("💣 Agora o desastre:")
gd("reset", "--hard", "HEAD~3")
gd("log", "--oneline", "-2")

In [ ]:
# SOLUÇÃO: reflog
gd("reflog", "-6")

In [ ]:
r = subprocess.run(["git", "reflog", "--format=%H %gs"], cwd=D, capture_output=True, text=True)
linhas = r.stdout.strip().splitlines()
alvo = None
for i, linha in enumerate(linhas):
    if "reset: moving to HEAD~3" in linha:
        alvo = linhas[i + 1].split()[0]
        break

gd("reset", "--hard", alvo)
gd("log", "--oneline", "-4")
print("✅ Os 3 commits voltaram.")

In [ ]:
# ══════════════════════════════════════════════════════════
# DESASTRE 4 — "Quebrei a produção com o último deploy"
# ══════════════════════════════════════════════════════════
gd("switch", "main", mostrar=False)
ed("calculo.py", '''def calcular_total(itens):
    """BUG: multiplica quando deveria somar."""
    total = 1
    for i in itens:
        total *= i["valor"]
    return total
''')
gd("add", ".", mostrar=False)
gd("commit", "-m", "perf: otimiza cálculo de total", mostrar=False)
print("😱 Já foi para produção (push feito). Não dá para usar reset.")
gd("log", "--oneline", "-2")

In [ ]:
# SOLUÇÃO: revert — seguro para histórico compartilhado
gd("revert", "HEAD", "--no-edit")
gd("log", "--oneline", "-3")
print(">>> O arquivo com bug ainda existe?", (D / "calculo.py").exists())
print("✅ Produção restaurada, histórico preservado, ninguém precisa re-clonar.")

In [ ]:
# ══════════════════════════════════════════════════════════
# DESASTRE 5 — "Meu PR tem 8 commits de 'wip'"
# ══════════════════════════════════════════════════════════
gd("switch", "-c", "feature/relatorio-csv", mostrar=False)
for i, msg in enumerate(["wip", "wip2", "tentando", "agora vai", "aff", "consertando", "ok?", "pronto"], 1):
    ed("exportador.py", f"# versão {i}\ndef exportar_csv(dados):\n    pass\n")
    gd("add", ".", mostrar=False)
    gd("commit", "-m", msg, mostrar=False)

print("😬 O que o revisor vai ver:")
gd("log", "--oneline", "-9")

In [ ]:
# SOLUÇÃO: reset --soft junta tudo (o branch é SEU, ainda não foi enviado)
gd("reset", "--soft", "HEAD~8")
gd(
    "commit",
    "-m", "feat(exportador): adiciona exportação de relatório em CSV",
    "-m", "Permite que a área comercial abra os dados no Excel sem\ndepender do time de engenharia.",
    mostrar=False,
)
print("✅ Um commit limpo:")
gd("log", "--oneline", "-3")

## 📝 Exercícios rápidos

**E1.** Modifique um arquivo rastreado, confirme com `git diff`, e descarte com `git restore`. Depois faça o mesmo mas dando `add` antes — qual comando você precisa agora?

**E2.** Faça um commit com a mensagem `"asdf"`. Corrija com `--amend`. Compare os hashes antes e depois e explique o que aconteceu.

**E3.** Faça 4 commits pequenos e junte-os em um só com `reset --soft`. Confirme que nenhuma linha de código se perdeu.

**E4.** Crie um commit que "quebra" algo, e desfaça com `revert`. Depois olhe o `git log` e explique por que o histórico ficou com **dois** commits em vez de zero.

**E5.** Comece uma alteração, dê `stash -u`, troque de branch, volte e dê `pop`. O que acontece se você der `stash` duas vezes seguidas? Como aplicar o primeiro?

**E6.** Faça 3 commits, apague-os com `reset --hard`, e recupere-os pelo `reflog`. Cronometre — em quanto tempo você consegue?

**E7.** Crie um branch com 3 commits e traga **apenas o do meio** para `main` com `cherry-pick`.

**E8.** Para cada situação, diga qual comando usar e **por quê**:
   - a) Commitei em `main` o que devia estar num branch (ainda não dei push)
   - b) Fiz push de um commit que quebrou a produção
   - c) Preciso trocar de branch mas tenho trabalho pela metade
   - d) Adicionei ao stage um arquivo que não devia
   - e) Quero ver o conteúdo de um arquivo como ele era há 5 commits

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

<!-- E8 — escreva suas respostas aqui -->

## 📋 Cola de referência

```bash
# ── Desfazer arquivos ──
git restore arquivo                    # descarta edição
git restore --staged arquivo           # tira do stage
git restore --staged --worktree arq    # ambos
git restore --source=HEAD~3 arquivo    # versão de 3 commits atrás
git clean -n                           # simula limpeza de untracked
git clean -f                           # apaga untracked
git clean -fd                          # apaga untracked + pastas

# ── Corrigir o último commit (só se NÃO enviado) ──
git commit --amend -m "nova mensagem"
git commit --amend --no-edit           # só adiciona arquivos ao commit

# ── Desfazer commits ──
git revert <hash>                      # ✅ seguro, cria commit inverso
git revert HEAD --no-edit
git revert -m 1 <merge>                # desfaz um merge
git reset --soft HEAD~3                # junta commits
git reset HEAD~1                       # desfaz commit, mantém arquivos
git reset --hard HEAD~1                # 💣 apaga commit E arquivos

# ── Guardar temporariamente ──
git stash -u -m "descrição"
git stash list
git stash show -p
git stash pop                          # aplica e remove
git stash apply stash@{1}              # aplica um específico
git stash branch nome-do-branch        # vira um branch

# ── Recuperar ──
git reflog                             # 🆘 todo movimento do HEAD
git reset --hard HEAD@{3}
git switch -c recuperacao HEAD@{3}     # mais seguro que reset
git fsck --lost-found                  # commits realmente órfãos

# ── Trazer commits ──
git cherry-pick <hash>
git cherry-pick <h1> <h2>
git cherry-pick --abort

# ── Cancelar operações ──
git merge --abort
git revert --abort
git cherry-pick --abort
git rebase --abort
```

## ✅ Checklist de saída

- [ ] Sei responder "onde está a coisa que quero desfazer?" antes de escolher o comando
- [ ] Uso `git restore` para arquivos e `git reset` para commits
- [ ] Sei que `--amend` cria um commit novo, e só uso antes do push
- [ ] Escolho `revert` para o que já foi enviado e `reset` para o que é só meu
- [ ] Conheço a diferença entre `--soft`, `--mixed` e `--hard`
- [ ] Sei que `--hard` destrói trabalho não commitado
- [ ] Uso `git stash` para trocar de contexto sem commitar lixo
- [ ] Rodo `git clean -n` antes de `git clean -f`
- [ ] Sei que o `reflog` existe e como usá-lo sob pressão
- [ ] Entendo que o reflog não salva o que nunca virou commit
- [ ] Sei que segredo commitado e enviado = segredo comprometido

---

### ➡️ Próxima etapa

**`02_99_Lista_Exercicios.ipynb`** — prática guiada com cenários completos, e a evolução do `projeto_Atlas` para o Módulo 02.